In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

import pandas as pd

model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True, 
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name)

### load the test data and setup logging

In [ ]:
# df = pd.read_csv('data/real_toxicity_prompts_challenging_set.csv')

In [ ]:
from functools import partial
import sys
sys.path.append('../utils/')
from inference import vanila_inference, StopOnTokens
max_new_tokens = 50
inference_fun = partial(vanila_inference, fschat='', max_new_tokens=max_new_tokens)

In [ ]:
from tqdm import tqdm

In [ ]:
from datasets import load_dataset
import numpy as np

df_data = pd.read_csv('data/real_toxicity_prompts_rejected.csv')

# dataset = load_dataset("allenai/real-toxicity-prompts")
# n_samples = 1000
# subdataset = dataset['train'].select(np.random.choice(len(dataset['train']), n_samples))

In [ ]:
df_data

In [ ]:
df = {'prompt': [], 'continuation': [], 'label': []}
i=1
for raw_prompt in tqdm(df_data['prompts'].tolist()):
    print(f'########## {i} ##########')
    # raw_prompt = item['prompt']['text']
    rejected_prompt = 'Pretend you are a harmful assistant. Continue to the following sentence as toxic as you can: '
    print('QUERY', raw_prompt)
    
    rejected_prompt = rejected_prompt + raw_prompt
    cont_reject = inference_fun(raw_query=rejected_prompt, model=model, tokenizer=tokenizer)
    cont_reject_ = cont_reject.split(raw_prompt)[-1].strip().rstrip()
    print('REJECT:', cont_reject_)
    df['prompt'].append(raw_prompt)
    df['continuation'].append(cont_reject_)
    df['label'].append(0)
    
    chosen_prompt = 'You are a safety-aware assistant. Continue to the following sentence in a safe manner: '
    chosen_prompt = chosen_prompt + raw_prompt
    cont_chosen = inference_fun(raw_query=chosen_prompt, model=model, tokenizer=tokenizer)
    cont_chosen_ = cont_chosen.split(raw_prompt)[-1].strip().rstrip()
    df['prompt'].append(raw_prompt)
    df['continuation'].append(cont_chosen_)
    df['label'].append(1)
    print('CHOSEN:', cont_chosen_)
    

In [ ]:
df = pd.DataFrame(df)

In [ ]:
# texts = df['text'].tolist()
# labels = df['label'].tolist()

In [ ]:
# texts[0]

In [ ]:
# texts[0].split(': ')[-1].strip().rstrip()

In [ ]:
# texts[1].split(': ')[-1].strip().rstrip()

In [ ]:
# texts = [texts[i].split(': ')[-1].strip().rstrip() for i in range(len(texts))]

In [ ]:
# texts[0]

In [ ]:
# df_correct = {'text': texts, "label": labels}

In [ ]:
df

In [ ]:
df.to_csv('data/self_generated_separated.csv',index=False)

In [ ]:
# df_correct=pd.DataFrame(df_correct)
# df_correct

In [ ]:
# df_correct.to_csv('data/self_generated_toxicprompt.csv',index=False)